# RQ4 (Portfolio Examination)

## From Scratch to Pretrained -- When Does Pretraining Pay Off?
#### *Comparing LSTM, BERT Fine-Tuning, and LLM Prompting Across Dataset Sizes*

**Due Date:** May 26, 2026
**Team:** NW 2
**Members:** Nils Wagner, Nick Wenzel

---

### Research Question

> *How do a from-scratch LSTM, a fine-tuned BERT model, and a prompted LLM compare on sentiment classification — and at what dataset size does pretraining become worthwhile?*

### What This Notebook Does

This template provides:
1. IMDB data loading with **configurable subset sizes** (50, 200, 1K, 5K, full)
2. **LSTM classifier** from Week 6 (working out of the box)
3. **BERT fine-tuning** pipeline with HuggingFace Trainer
4. **LLM prompting** pipeline (zero-shot and few-shot)
5. Experiment loop over all (approach × dataset size) combinations
6. Plotting helpers for the key accuracy-vs-size figure

**Your job:** Run the experiments, analyze the results, find the crossover points, and write the report.

Look for `# TODO` comments — these mark where you need to add your analysis.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re
import time

torch.manual_seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

# HuggingFace imports
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, pipeline
from datasets import load_dataset
import evaluate

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print("Setup complete.")

---
## Part 1: Data Loading with Configurable Subset Sizes

The key variable: **how many labeled training examples are available?**
We test with 50, 200, 1,000, 5,000, and the full 25,000.

In [ ]:
# ── Load IMDB dataset ──────────────────────────────────────────
dataset = load_dataset("imdb")
print(f"Full dataset: Train {len(dataset['train']):,}, Test {len(dataset['test']):,}")

# ── Function to create balanced subsets ────────────────────────
def get_subset(dataset, n_train, seed=42):
    """Sample n_train examples (balanced) from the training set."""
    if n_train >= len(dataset['train']):
        return dataset['train']
    # Balance: equal positive and negative
    pos = [i for i, ex in enumerate(dataset['train']) if ex['label'] == 1]
    neg = [i for i, ex in enumerate(dataset['train']) if ex['label'] == 0]
    rng = np.random.RandomState(seed)
    pos_sample = rng.choice(pos, n_train // 2, replace=False)
    neg_sample = rng.choice(neg, n_train // 2, replace=False)
    indices = np.concatenate([pos_sample, neg_sample])
    return dataset['train'].select(indices)

# Test it
for n in [50, 200, 1000, 5000]:
    subset = get_subset(dataset, n)
    labels = [ex['label'] for ex in subset]
    print(f"  n={n:5d}: {len(subset)} examples, {sum(labels)} pos, {len(labels)-sum(labels)} neg")

TRAIN_SIZES = [50, 200, 1000, 5000, 25000]
N_SEEDS = 3  # repeat each experiment with 3 different random seeds
print(f"\nExperiment plan: {len(TRAIN_SIZES)} sizes × {N_SEEDS} seeds = {len(TRAIN_SIZES)*N_SEEDS} runs per approach")

---
## Part 2: Model 1 — LSTM (From Scratch)

Reuses the LSTM architecture from Week 6 / RQ3. No pretrained embeddings — learns everything from the labeled training data.

In [ ]:
# ── Text pipeline for LSTM ─────────────────────────────────────
def tokenize(text):
    text = text.lower()
    text = re.sub(r'<br\s*/?>', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text.split()

def build_vocab(texts, max_vocab=20000, min_freq=2):
    counter = Counter()
    for text in texts:
        counter.update(tokenize(text))
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, count in counter.most_common(max_vocab - 2):
        if count >= min_freq:
            vocab[word] = len(vocab)
    return vocab

def encode(text, vocab, max_len=200):
    tokens = tokenize(text)[:max_len]
    ids = [vocab.get(t, vocab['<UNK>']) for t in tokens]
    ids = ids + [0] * (max_len - len(ids))
    return ids

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=200):
        self.encodings = [encode(t, vocab, max_len) for t in texts]
        self.labels = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return (torch.tensor(self.encodings[idx], dtype=torch.long),
                torch.tensor(self.labels[idx], dtype=torch.long))

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128, num_classes=2, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)
    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        return self.fc(self.dropout(h_n.squeeze(0)))

print("LSTM pipeline defined.")

In [ ]:
# ── Train and evaluate LSTM ─────────────────────────────────────
def train_lstm(train_subset, test_dataset, vocab, epochs=10, lr=1e-3):
    """Train LSTM on a subset, evaluate on full test set."""
    train_texts = [ex['text'] for ex in train_subset]
    train_labels = [ex['label'] for ex in train_subset]

    train_ds = SentimentDataset(train_texts, train_labels, vocab)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

    model = LSTMClassifier(len(vocab)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
    train_time = time.time() - t0

    # Evaluate on full test set
    model.eval()
    test_loader = DataLoader(test_dataset, batch_size=128)
    correct, total = 0, 0
    with torch.no_grad():
        for bx, by in test_loader:
            bx, by = bx.to(device), by.to(device)
            correct += (model(bx).argmax(1) == by).sum().item()
            total += by.size(0)

    return correct / total, train_time

# Build vocab from FULL training set (so it's consistent across subset sizes)
all_train_texts = [ex['text'] for ex in dataset['train']]
vocab = build_vocab(all_train_texts)
print(f"Vocabulary: {len(vocab):,} words")

# Prepare full test set (used for all evaluations)
test_texts = [ex['text'] for ex in dataset['test']]
test_labels = [ex['label'] for ex in dataset['test']]
test_ds = SentimentDataset(test_texts, test_labels, vocab)
print(f"Test set: {len(test_ds):,} examples (used for ALL evaluations)")

---
## Part 2b: Model 2 — LSTM with GloVe Embeddings (Static Pretraining)

Same LSTM architecture, but with pretrained GloVe word vectors instead of random embeddings. This tests: **how much of the LSTM's weakness in RQ3 was due to learning embeddings from scratch?**

In [ ]:
# ── Load GloVe embeddings ──────────────────────────────────────
import gensim.downloader as api

print("Loading GloVe embeddings (this may take a moment on first run)...")
try:
    glove = api.load("glove-wiki-gigaword-100")  # 100d GloVe vectors
    GLOVE_DIM = 100
    print(f"Loaded GloVe: {len(glove.key_to_index):,} words, {GLOVE_DIM} dimensions")
except Exception as e:
    print(f"Could not load GloVe via gensim: {e}")
    print("Falling back to random embeddings for the GloVe condition.")
    glove = None
    GLOVE_DIM = 100

In [ ]:
# ── Build GloVe embedding matrix for our vocabulary ────────────
def build_glove_matrix(vocab, glove_model, embed_dim=100):
    """Create an embedding matrix from GloVe for our vocabulary."""
    matrix = np.random.randn(len(vocab), embed_dim) * 0.01  # small random for OOV
    found, total = 0, 0
    for word, idx in vocab.items():
        total += 1
        if glove_model is not None and word in glove_model:
            matrix[idx] = glove_model[word]
            found += 1
    print(f"GloVe coverage: {found}/{total} words ({found/total:.1%})")
    return torch.tensor(matrix, dtype=torch.float32)

glove_matrix = build_glove_matrix(vocab, glove)

# ── LSTM with frozen GloVe embeddings ──────────────────────────
def train_lstm_glove(train_subset, test_dataset, vocab, glove_matrix,
                     epochs=10, lr=1e-3):
    """Train LSTM with frozen GloVe embeddings."""
    train_texts = [ex['text'] for ex in train_subset]
    train_labels = [ex['label'] for ex in train_subset]

    train_ds = SentimentDataset(train_texts, train_labels, vocab)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

    model = LSTMClassifier(len(vocab), embed_dim=GLOVE_DIM).to(device)

    # Initialize with GloVe and freeze
    model.embedding.weight.data.copy_(glove_matrix)
    model.embedding.weight.requires_grad = False

    # Only optimize non-embedding parameters
    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=lr)
    criterion = nn.CrossEntropyLoss()

    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
    train_time = time.time() - t0

    model.eval()
    test_loader = DataLoader(test_dataset, batch_size=128)
    correct, total = 0, 0
    with torch.no_grad():
        for bx, by in test_loader:
            bx, by = bx.to(device), by.to(device)
            correct += (model(bx).argmax(1) == by).sum().item()
            total += by.size(0)

    return correct / total, train_time

print("LSTM + GloVe pipeline defined.")

---
## Part 2c: Model 3 — BiLSTM + GloVe + Attention Pooling

Same GloVe embeddings as Model 2, but with a **bidirectional LSTM** and **attention pooling** over all hidden states. This isolates the effect of better architecture and aggregation.

In [ ]:
# ── BiLSTM with attention pooling ───────────────────────────────

class AttentionPool(nn.Module):
    """Learned attention pooling over LSTM hidden states."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
    def forward(self, lstm_output):
        scores = self.attn(lstm_output).squeeze(-1)  # (B, L)
        weights = F.softmax(scores, dim=1)            # (B, L)
        return (weights.unsqueeze(-1) * lstm_output).sum(dim=1)  # (B, H)

class BiLSTMAttnClassifier(nn.Module):
    """BiLSTM with GloVe embeddings and attention pooling."""
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128, num_classes=2,
                 dropout=0.5, pretrained_embeddings=None, freeze_emb=False):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if pretrained_embeddings is not None:
            self.embedding.weight.data.copy_(pretrained_embeddings)
        if freeze_emb:
            self.embedding.weight.requires_grad = False
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.attention = AttentionPool(hidden_dim * 2)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
    def forward(self, x):
        x = self.embedding(x)
        lstm_out, _ = self.lstm(x)
        pooled = self.attention(lstm_out)
        return self.fc(self.dropout(pooled))

def train_bilstm_glove(train_subset, test_dataset, vocab, glove_matrix,
                       epochs=10, lr=1e-3):
    """Train BiLSTM+attention with frozen GloVe embeddings."""
    train_texts = [ex['text'] for ex in train_subset]
    train_labels = [ex['label'] for ex in train_subset]
    train_ds = SentimentDataset(train_texts, train_labels, vocab)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

    model = BiLSTMAttnClassifier(
        len(vocab), embed_dim=GLOVE_DIM,
        pretrained_embeddings=glove_matrix, freeze_emb=True
    ).to(device)

    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=lr)
    criterion = nn.CrossEntropyLoss()

    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
    train_time = time.time() - t0

    model.eval()
    test_loader = DataLoader(test_dataset, batch_size=128)
    correct, total = 0, 0
    with torch.no_grad():
        for bx, by in test_loader:
            bx, by = bx.to(device), by.to(device)
            correct += (model(bx).argmax(1) == by).sum().item()
            total += by.size(0)
    return correct / total, train_time

print("BiLSTM + GloVe + AttentionPool defined.")
print("  Compared to Model 2: adds bidirectionality + attention pooling")
print("  Compared to Model 1: adds GloVe + bidirectionality + attention pooling")

---
## Part 3: Model 4 — BERT Fine-Tuning

Fine-tune DistilBERT (faster than full BERT, 97% of the quality) using HuggingFace.

In [ ]:
# ── BERT fine-tuning pipeline ──────────────────────────────────
BERT_MODEL = "distilbert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return accuracy_metric.compute(predictions=np.argmax(logits, axis=-1), references=labels)

def train_bert(train_subset, n_train, epochs=3, lr=2e-5):
    """Fine-tune BERT on a subset, evaluate on full test set."""
    # Tokenize train subset
    train_texts = [ex['text'] for ex in train_subset]
    train_labels = [ex['label'] for ex in train_subset]

    train_enc = bert_tokenizer(train_texts, truncation=True, padding="max_length",
                                max_length=256, return_tensors="pt")
    train_enc['labels'] = torch.tensor(train_labels)

    # Tokenize test set (first 2000 for speed in template; use full for final results)
    test_sample = dataset['test'].select(range(min(2000, len(dataset['test']))))
    test_texts_b = [ex['text'] for ex in test_sample]
    test_labels_b = [ex['label'] for ex in test_sample]
    test_enc = bert_tokenizer(test_texts_b, truncation=True, padding="max_length",
                               max_length=256, return_tensors="pt")
    test_enc['labels'] = torch.tensor(test_labels_b)

    # Create simple datasets
    class SimpleDataset(Dataset):
        def __init__(self, encodings):
            self.encodings = encodings
        def __len__(self): return len(self.encodings['labels'])
        def __getitem__(self, idx):
            return {k: v[idx] for k, v in self.encodings.items()}

    # Fresh model for each run
    model = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL, num_labels=2)

    args = TrainingArguments(
        output_dir=f"./tmp_rq4_{n_train}",
        num_train_epochs=epochs,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=lr,
        weight_decay=0.01,
        eval_strategy="no",
        save_strategy="no",
        logging_steps=9999,  # suppress logs
        report_to="none",
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(model=model, args=args,
                      train_dataset=SimpleDataset(train_enc),
                      eval_dataset=SimpleDataset(test_enc),
                      compute_metrics=compute_metrics)

    t0 = time.time()
    trainer.train()
    train_time = time.time() - t0

    results = trainer.evaluate()
    return results['eval_accuracy'], train_time

print("BERT fine-tuning pipeline defined.")
print(f"Using: {BERT_MODEL}")

---
## Part 4: Model 5 — LLM Prompting

Classify sentiment using prompting — no training, no weight updates. The model classifies based on the prompt alone.

In [ ]:
# ── Prompting pipeline ─────────────────────────────────────────
# Use a text-generation model for prompting
# For better results, use a larger model or an API

PROMPT_MODEL = "gpt2"  # Small, runs on CPU. For better results: gpt2-medium, or API

prompt_generator = pipeline("text-generation", model=PROMPT_MODEL,
                            device=device, max_new_tokens=5)

def classify_with_prompt(text, examples=None):
    """Classify sentiment using zero-shot or few-shot prompting."""
    if examples:
        # Few-shot: include examples
        prompt_parts = ["Classify sentiment as Positive or Negative.\n"]
        for ex_text, ex_label in examples:
            label_str = "Positive" if ex_label == 1 else "Negative"
            prompt_parts.append(f'Review: "{ex_text[:100]}" → {label_str}')
        prompt_parts.append(f'Review: "{text[:200]}" →')
        prompt = "\n".join(prompt_parts)
    else:
        # Zero-shot
        prompt = f'Classify the sentiment as Positive or Negative.\nReview: "{text[:200]}"\nSentiment:'

    try:
        result = prompt_generator(prompt, pad_token_id=prompt_generator.tokenizer.eos_token_id)[0]
        generated = result['generated_text'][len(prompt):].strip().lower()
        if 'positive' in generated[:20]:
            return 1
        elif 'negative' in generated[:20]:
            return 0
        else:
            return -1  # couldn't parse
    except Exception:
        return -1

def evaluate_prompting(test_texts, test_labels, examples=None, max_test=200):
    """Evaluate prompting on a sample of the test set."""
    correct, total, unparsed = 0, 0, 0
    t0 = time.time()
    for text, label in zip(test_texts[:max_test], test_labels[:max_test]):
        pred = classify_with_prompt(text, examples)
        if pred == -1:
            unparsed += 1
        elif pred == label:
            correct += 1
        total += 1

    elapsed = time.time() - t0
    valid = total - unparsed
    acc = correct / valid if valid > 0 else 0
    return acc, elapsed, unparsed

print(f"Prompting pipeline defined. Using: {PROMPT_MODEL}")
print(f"Note: for better prompting results, use gpt2-medium or an API.")

---
## Part 5: Run All Experiments

The core experiment loop: 3 approaches × 5+ dataset sizes × 3 seeds.

**Warning:** This takes a while! BERT fine-tuning is the bottleneck.
On CPU: ~1-2 hours total. On GPU: ~20-30 minutes.

In [ ]:
# ── Experiment loop ────────────────────────────────────────────
results = {}

# ── LSTM experiments ──────────────────────────────────────────
print("=" * 60)
print("LSTM EXPERIMENTS")
print("=" * 60)

for n_train in TRAIN_SIZES:
    accs = []
    for seed in range(N_SEEDS):
        subset = get_subset(dataset, n_train, seed=42 + seed)
        acc, t = train_lstm(subset, test_ds, vocab, epochs=10)
        accs.append(acc)
        print(f"  n={n_train:5d}, seed={seed}: acc={acc:.4f}, time={t:.1f}s")
    results[('LSTM', n_train)] = {'mean': np.mean(accs), 'std': np.std(accs)}
    print(f"  → LSTM n={n_train}: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print()

In [ ]:
# ── LSTM + GloVe experiments ───────────────────────────────────
print("=" * 60)
print("LSTM + GloVe EXPERIMENTS")
print("=" * 60)

for n_train in TRAIN_SIZES:
    accs = []
    for seed in range(N_SEEDS):
        subset = get_subset(dataset, n_train, seed=42 + seed)
        acc, t = train_lstm_glove(subset, test_ds, vocab, glove_matrix, epochs=10)
        accs.append(acc)
        print(f"  n={n_train:5d}, seed={seed}: acc={acc:.4f}, time={t:.1f}s")
    results[('LSTM+GloVe', n_train)] = {'mean': np.mean(accs), 'std': np.std(accs)}
    print(f"  → LSTM+GloVe n={n_train}: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print()

In [ ]:
# ── BiLSTM + GloVe + Attention experiments ─────────────────────
print("=" * 60)
print("BiLSTM + GloVe + ATTENTION EXPERIMENTS")
print("=" * 60)

for n_train in TRAIN_SIZES:
    accs = []
    for seed in range(N_SEEDS):
        subset = get_subset(dataset, n_train, seed=42 + seed)
        acc, t = train_bilstm_glove(subset, test_ds, vocab, glove_matrix, epochs=10)
        accs.append(acc)
        print(f"  n={n_train:5d}, seed={seed}: acc={acc:.4f}, time={t:.1f}s")
    results[('BiLSTM+Attn', n_train)] = {'mean': np.mean(accs), 'std': np.std(accs)}
    print(f"  → BiLSTM+Attn n={n_train}: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print()

In [ ]:
# ── BERT experiments ───────────────────────────────────────────
print("=" * 60)
print("BERT EXPERIMENTS")
print("=" * 60)

for n_train in TRAIN_SIZES:
    accs = []
    for seed in range(N_SEEDS):
        subset = get_subset(dataset, n_train, seed=42 + seed)
        acc, t = train_bert(subset, n_train, epochs=3)
        accs.append(acc)
        print(f"  n={n_train:5d}, seed={seed}: acc={acc:.4f}, time={t:.1f}s")
    results[('BERT', n_train)] = {'mean': np.mean(accs), 'std': np.std(accs)}
    print(f"  → BERT n={n_train}: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print()

In [ ]:
# ── Prompting experiments ──────────────────────────────────────
print("=" * 60)
print("PROMPTING EXPERIMENTS")
print("=" * 60)

prompt_test_texts = [ex['text'] for ex in dataset['test']]
prompt_test_labels = [ex['label'] for ex in dataset['test']]

# Zero-shot
acc_zs, t_zs, unp = evaluate_prompting(prompt_test_texts, prompt_test_labels, max_test=200)
results[('Prompt-ZS', 0)] = {'mean': acc_zs, 'std': 0.0}
print(f"  Zero-shot: acc={acc_zs:.4f}, unparsed={unp}, time={t_zs:.1f}s")

# Few-shot with k examples
for k in [2, 4, 8]:
    accs = []
    for seed in range(N_SEEDS):
        rng = np.random.RandomState(42 + seed)
        # Sample k examples from training set
        pos = [i for i, ex in enumerate(dataset['train']) if ex['label'] == 1]
        neg = [i for i, ex in enumerate(dataset['train']) if ex['label'] == 0]
        ex_idx = list(rng.choice(pos, k//2, replace=False)) + list(rng.choice(neg, k//2, replace=False))
        examples = [(dataset['train'][i]['text'], dataset['train'][i]['label']) for i in ex_idx]

        acc, t, _ = evaluate_prompting(prompt_test_texts, prompt_test_labels, examples, max_test=200)
        accs.append(acc)
        print(f"  Few-shot k={k}, seed={seed}: acc={acc:.4f}")

    results[(f'Prompt-{k}shot', k)] = {'mean': np.mean(accs), 'std': np.std(accs)}
    print(f"  → Prompt k={k}: {np.mean(accs):.4f} ± {np.std(accs):.4f}")

print("\nAll experiments complete!")

---
## Part 6: The Key Plot — Accuracy vs. Dataset Size

In [ ]:
# ── The central figure ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 7))

# LSTM (random) curve
lstm_sizes = TRAIN_SIZES
lstm_accs = [results[('LSTM', n)]['mean'] for n in lstm_sizes]
lstm_stds = [results[('LSTM', n)]['std'] for n in lstm_sizes]
ax.errorbar(lstm_sizes, lstm_accs, yerr=lstm_stds, fmt='o-', color='#E85D75',
            linewidth=2, markersize=8, capsize=4, label='LSTM (random emb.)')

# LSTM (GloVe) curve
glove_accs = [results[('LSTM+GloVe', n)]['mean'] for n in lstm_sizes]
glove_stds = [results[('LSTM+GloVe', n)]['std'] for n in lstm_sizes]
ax.errorbar(lstm_sizes, glove_accs, yerr=glove_stds, fmt='^-', color='#E8A838',
            linewidth=2, markersize=8, capsize=4, label='LSTM (GloVe emb.)')

# BiLSTM + GloVe + Attention curve
bi_accs = [results[('BiLSTM+Attn', n)]['mean'] for n in lstm_sizes]
bi_stds = [results[('BiLSTM+Attn', n)]['std'] for n in lstm_sizes]
ax.errorbar(lstm_sizes, bi_accs, yerr=bi_stds, fmt='D-', color='#8B5CF6',
            linewidth=2, markersize=8, capsize=4, label='BiLSTM+GloVe+Attn')

# BERT curve
bert_accs = [results[('BERT', n)]['mean'] for n in lstm_sizes]
bert_stds = [results[('BERT', n)]['std'] for n in lstm_sizes]
ax.errorbar(lstm_sizes, bert_accs, yerr=bert_stds, fmt='s-', color='#27AE60',
            linewidth=2, markersize=8, capsize=4, label='BERT (fine-tuned)')

# Prompting horizontal lines
if ('Prompt-ZS', 0) in results:
    ax.axhline(y=results[('Prompt-ZS', 0)]['mean'], color='#4A90D9',
               linestyle='--', linewidth=2, label=f"Zero-shot prompting ({PROMPT_MODEL})")

for k in [2, 4, 8]:
    key = (f'Prompt-{k}shot', k)
    if key in results:
        ax.axhline(y=results[key]['mean'], color='#8B5CF6',
                   linestyle=':', linewidth=1.5, alpha=0.6,
                   label=f"{k}-shot prompting")

ax.set_xscale('log')
ax.set_xlabel("Number of Training Examples (log scale)", fontsize=13)
ax.set_ylabel("Test Accuracy", fontsize=13)
ax.set_title("RQ4: When Does Pretraining Pay Off?", fontweight='bold', fontsize=14)
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_ylim(0.5, 1.0)
ax.set_xticks(TRAIN_SIZES)
ax.set_xticklabels([str(n) for n in TRAIN_SIZES])

plt.tight_layout()
plt.show()

# TODO: Identify the crossover points in your report:
# - How much does GloVe help the LSTM? Is the gap constant or does it shrink?
# - At what size does BERT overtake both LSTM variants?
# - At what size does BERT overtake few-shot prompting?
# - What is the "pretraining gradient": random → GloVe → BERT?

### Results Table

In [ ]:
# ── Print results table ────────────────────────────────────────
print(f"{'Approach':<20} {'N':>6} {'Accuracy':>12} {'± Std':>8}")
print("-" * 50)

for approach in ['LSTM', 'LSTM+GloVe', 'BiLSTM+Attn', 'BERT']:
    for n in TRAIN_SIZES:
        r = results.get((approach, n))
        if r:
            print(f"{approach:<20} {n:>6} {r['mean']:>12.4f} {r['std']:>8.4f}")

for key in [('Prompt-ZS', 0), ('Prompt-2shot', 2), ('Prompt-4shot', 4), ('Prompt-8shot', 8)]:
    r = results.get(key)
    if r:
        print(f"{key[0]:<20} {key[1]:>6} {r['mean']:>12.4f} {r['std']:>8.4f}")

# TODO: Include this table in your report

---
## Part 7: Error Analysis

Pick one dataset size (e.g., n=200) and compare what each approach gets right and wrong.

`# TODO`: This is the most important analytical section.

In [ ]:
# TODO: Error analysis
# 1. Train LSTM and BERT on n=200 examples
# 2. Run prompting (zero-shot and few-shot)
# 3. Find examples where:
#    - BERT correct, LSTM wrong (pretraining helps)
#    - Prompting correct, both trained models wrong (LLM knowledge helps)
#    - All wrong (hard examples)
# 4. Show 3-5 concrete examples per category
# 5. Explain WHY each approach succeeds or fails

print("TODO: Add your error analysis here")

---
## Part 8: Your Analysis

`# TODO`: Add your own analysis cells. Consider:

1. **The pretraining gradient:** How much does each level (random → GloVe → BERT) improve accuracy?
2. **GloVe impact:** Does GloVe help more at small sizes than large? (Expected: yes — the prior matters more when data is scarce.)
3. **The crossover point:** At what dataset size does BERT overtake both LSTM variants?
4. **Prompting vs fine-tuning:** Does fine-tuning with 50 examples beat few-shot prompting?
5. **Cost-benefit analysis:** Training time (LSTM, BERT) vs inference cost (prompting per example)
6. **Practical recommendation:** Given N labeled examples, which approach should a practitioner choose?

These questions should inform your report discussion.

In [ ]:
# TODO: Your additional analysis
# Ideas:
# - Plot training time vs accuracy
# - Compare inference speed: LSTM vs BERT vs prompting per example
# - Test with different prompt templates
# - Compare DistilBERT vs full BERT at small sizes


---
## Summary

Fill in your key findings:

| Finding | Evidence |
|---------|----------|
| **Gap 1→2** (GloVe effect) at n=200 | [your result] |
| **Gap 2→3** (BiLSTM+attn effect) at n=200 | [your result] |
| **Gap 3→4** (contextual pretraining) at n=200 | [your result] |
| BiLSTM+GloVe+Attn vs TextCNN (RQ3) | [your result] |
| BERT overtakes few-shot prompting at n = ? | [your result] |
| BERT overtakes few-shot prompting at n = ? | [your result] |
| Zero-shot prompting accuracy | [your result] |
| Best overall approach at n=200 | [your result] |
| Training time: LSTM vs BERT | [your result] |

### Practical Recommendation

*[Based on your results: if a practitioner has N labeled examples, which approach should they use? Consider accuracy, training cost, and inference speed.]*

### Key Takeaway

*[Write 2-3 sentences summarizing the most important finding. When is pretraining worth the effort?]*